# NeuroGraph-X · Week 5 · Tuning grid (memo 0011), all four atlases
Owner: Laiba Afridi. **Cross-validation only: this notebook never builds a frozen-set graph and has no final cell.**

48 runs = 12 configurations (edge density 0.05 / 0.10 / 0.20 × class weights off / on × hidden 32 / 64) × 4 atlases, each one 5-fold CV on the 761 development participants, tagged `0011-v1`. Runs must be logged before the freeze (Monday 2 November 2026, 00:00 Pakistan time).

**How to run it without watching:** add the `neurographx-code` dataset, set GPU T4 and Internet On, then **Save Version → Save & Run All (Commit)**. Kaggle runs everything in the background (about 3 to 4 hours) and keeps the files in the version's Output. Every cell checks itself and stops the run if something is wrong.

If it stops part-way, open the notebook and run it again: finished configurations are skipped.

## 1. Copy the code into a writable folder

In [ ]:
import glob, os, shutil, subprocess, sys
code_root = os.path.dirname(glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)[0])
REPO = "/kaggle/working/ngx"
if not os.path.exists(REPO):
    shutil.copytree(code_root, REPO)
os.chdir(REPO)
print("repo:", REPO, "| commit:", open("COMMIT").read().strip() if os.path.exists("COMMIT") else "unknown")
ENV = {**os.environ, "PYTHONPATH": "src", "PYTHONUNBUFFERED": "1"}

def run(*cmd):
    """Run a command, show its output as it comes, stop the notebook if it fails."""
    print("$", " ".join(cmd), flush=True)
    done = subprocess.run(list(cmd), env=ENV)
    assert done.returncode == 0, f"failed with exit code {done.returncode}: {' '.join(cmd)}"

In [ ]:
!pip install -q torch_geometric==2.8.0.post1 nilearn==0.14.1 tabulate
import torch, torch_geometric; print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| GPU", torch.cuda.is_available())

## 2. Rebuild the connectomes (memo 0007). The download is resumable; about an hour.

In [ ]:
run(sys.executable, "scripts/01_fetch_phenotypic.py")
for attempt in range(3):  # the download can drop; each retry continues where it stopped
    if subprocess.run([sys.executable, "scripts/04_fetch_all_atlases.py", "--quiet"], env=ENV).returncode == 0:
        break
else:
    raise SystemExit("04_fetch_all_atlases.py failed three times")
run(sys.executable, "scripts/07_build_connectomes.py")
run(sys.executable, "scripts/08_verify_connectomes.py")

## 3. Check: the rebuilt matrices must match the committed report before any training

In [ ]:
expected = {  # key: (participants, nodes, flat-region participants, NaN edges), docs/reports/connectome_summary.md
    "rois_aal": (861, 116, 54, 2014), "rois_cc200": (861, 200, 15, 3969),
    "rois_ho": (861, 111, 198, 1055), "rois_dosenbach160": (861, 161, 46, 2440)}
rows = {}
for line in open("docs/reports/connectome_summary.md"):
    cells = [c.strip() for c in line.strip().strip("|").split("|")]
    if len(cells) > 6 and cells[1] in expected:
        rows[cells[1]] = (int(cells[2]), int(cells[3]), int(cells[5]), int(cells[6]))
for key, want in expected.items():
    print(f"{key:>18}: {rows.get(key)}  expected {want}")
    assert rows.get(key) == want, f"{key} differs from the committed report: stop and ask"
print("All four atlases match the committed report.")

## 4. Tests (all must pass)

In [ ]:
run(sys.executable, "-m", "pytest", "-q", "tests/test_leakage.py", "tests/test_gnn_data.py",
    "tests/test_run_logger.py", "tests/test_inference.py", "tests/test_oof.py", "tests/test_tuning.py")

## 5. The tuning grid, one atlas per cell (12 runs each, about 25 to 40 minutes per atlas)

In [ ]:
run(sys.executable, "scripts/run_tuning_grid.py", "--atlas", "rois_cc200")

In [ ]:
run(sys.executable, "scripts/run_tuning_grid.py", "--atlas", "rois_aal")

In [ ]:
run(sys.executable, "scripts/run_tuning_grid.py", "--atlas", "rois_ho")

In [ ]:
run(sys.executable, "scripts/run_tuning_grid.py", "--atlas", "rois_dosenbach160")

## 6. Count: 12 tagged runs per atlas (no scores here; selection is done by script after the freeze)

In [ ]:
import json
from collections import Counter
runs = [json.loads(l) for l in open("runs/run_log.jsonl") if l.strip()]
tagged = [r for r in runs if (r.get("data_config") or {}).get("tuning_grid") == "0011-v1"]
count = Counter(r["atlas"] for r in tagged)
print(dict(count), "| total", len(tagged), "| last logged", tagged[-1]["timestamp"] if tagged else None)
assert all(count[a] == 12 for a in ("aal", "cc200", "ho", "dosenbach160")), "some configurations are missing"

## 7. Evidence: the files to download

In [ ]:
os.makedirs("/kaggle/working/evidence/oof", exist_ok=True)
shutil.copy("runs/run_log.jsonl", "/kaggle/working/evidence/")
for f in glob.glob("runs/oof/*.csv"):
    shutil.copy(f, "/kaggle/working/evidence/oof/")
print(len(os.listdir("/kaggle/working/evidence/oof")), "OOF files;", sum(1 for _ in open("runs/run_log.jsonl")), "runs in the log")

Download `evidence/run_log.jsonl` and every file in `evidence/oof/` from the version's Output. There is no final cell: final models are Week 6, after selection (memo 0011).